# Spark-X2.6-4B-En4B-A1B (`v0.3`) — Continue Training from `TassP0lar/Spark-X2.6-4B-En4B-A1B` with Supervised Instructed Self-Recursive Self-Improvement (`SI-SRSI`) & Dynamic `MoE-Engram` Knowledge Formation

**Direct Continuation Pipeline from [`https://huggingface.co/TassP0lar/Spark-X2.6-4B-En4B-A1B`](https://huggingface.co/TassP0lar/Spark-X2.6-4B-En4B-A1B) (`72 Effective Layers`, `8.34B Total / 5.33B–6.13B Dynamic Active Parameters`)**

This standalone notebook loads your already-trained **`TassP0lar/Spark-X2.6-4B-En4B-A1B`** checkpoint directly from Hugging Face Hub—preserving all learned `72L TwoStreamMHC` and `32-Expert Top-7 ZipfianTieredMoEEngram` weights—and continues training to form specialized **Knowledge Engrams** via **Supervised Instructed Self-Recursive Self-Improvement (`SI-SRSI`)** supervised by an **OpenCode Teacher**:

1. **Domain-Partitioned Knowledge Engram Zones (`32` Routed Experts + `1` Shared Expert + `196,608` N-Gram Buckets per Anchor)**:
   - **Domain 1 (`anti_slop`, Experts `22..29` + Bridge `30..31`)**: Eliminates "Claude-ish" RLHF clichés (*"delve"*, *"tapestry"*, *"testament"*, *"not just X, but Y"*, sycophantic openers), enforces sentence-length burstiness ($	ext{CV} \ge 0.35$), and applies **Token-Level Anti-Slop Unlikelihood Regularization** (`FTPO` / Unlikelihood loss).
   - **Domain 2 (`web_ux_sec`, Experts `12..21` + Bridge `30..31`)**: Full-stack Frontend/Backend UX architecture, WCAG 2.2 AA accessibility (`aria-live`, keyboard focus, `prefers-reduced-motion`), 4-state UX policy (`loading`/`empty`/`error`/`success`), and **Defensive Application Security** (`Content-Security-Policy`, `X-Frame-Options: DENY`, `HttpOnly; Secure; SameSite=Strict` cookies, `X-CSRF-Token`, parameterized SQL `$1`, safe DOM `textContent` escaping, token-bucket rate limiting).
   - **Domain 3 (`stem_reason`, Experts `0..11` + Bridge `30..31`)**: First-principles STEM derivations, dimensional/conservation checks, and exact symbolic verification.
2. **Two Flexible Continuation Modes**:
   - **`CONTINUATION_MODE = "SI_SRSI_AND_PHASE4"` (Default, Fast & Targeted)**: Runs Stage A (Phase-4 Multi-Domain `MoE-Engram` + `mHC` Consolidation) + Stage B (`SI-SRSI` OpenCode-Supervised On-Policy Rollout $ightarrow$ Critique $ightarrow$ Domain-Partitioned Engram Consolidation with MGD Rollback Guard) + Stage C (On-the-Fly Flight Mode Validation).
   - **`CONTINUATION_MODE = "FULL_STREAMING_CORPUS_PLUS_SI_SRSI"` (Deep 4.11B Engram Expansion)**: Also streams fresh 7-Pillar 2025–2026 Frontier datasets (`SuperGPQA`, `OpenThoughts3-1.2M`, `NaturalReasoning`, `SWE-smith`, `UIGEN-T1.5`) + OpenCode Teacher trajectories and runs continued `8K`/`16K` `MoE-Engram-32` training on top of `TassP0lar/Spark-X2.6-4B-En4B-A1B` before Phase-4 and `SI-SRSI`.

In [ ]:
# ==============================================================================
# Cell 1: Clone / Sync Repository & Install Dependencies
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

pip_pkgs = [
    'torch>=2.5.0',
    'triton>=3.1.0',
    'transformers==4.57.1',
    'datasets>=3.2.0',
    'accelerate>=1.2.0',
    'safetensors>=0.4.5',
    'huggingface_hub>=0.27.0',
    'bitsandbytes>=0.45.0',
    'cut-cross-entropy',
    'liger-kernel',
    'unsloth',
    'openai>=1.58.0',
    'nltk',
    'langdetect',
    'psutil',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_pkgs], check=False)
print('Working directory synced:', Path.cwd())

In [ ]:
# ==============================================================================
# Cell 2: Load Existing Trained Checkpoint from `TassP0lar/Spark-X2.6-4B-En4B-A1B`
#         & Run Pre-Continuation Health / Perplexity / Router Baseline Audit
# ==============================================================================
import importlib, os, subprocess, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core
import sparkx_loop_v03_train_env
import sparkx_loop_v03_benchmarks as bench
import load_inference_v03
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
importlib.reload(bench)
importlib.reload(load_inference_v03)

from load_inference_v03 import load_sparkx_loop_v03
from sparkx_loop_v03_core import SparkXLoopV03, count_en4b_a1b_params

def _get_secret(name: str):
    val = os.environ.get(name)
    if val:
        return val.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return None

HF_TOKEN = _get_secret('HF_TOKEN')
OPENCODE_API_KEY = _get_secret('OPENCODE_API_KEY')
OPENCODE_API_URL = _get_secret('OPENCODE_API_URL')
HF_REPO_ID = 'TassP0lar/Spark-X2.6-4B-En4B-A1B'

if 'model' not in globals() or model is None:
    model, tokenizer = load_sparkx_loop_v03(
        repo_id=HF_REPO_ID,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        token=HF_TOKEN,
        enable_loglinear_kv=True,
    )
else:
    print(f"[VRAM Cache Hit] Reusing already-loaded `{HF_REPO_ID}` model in GPU memory.")

# Hot-bind latest SI-SRSI & Dynamic Flight Mode methods onto live model instance
type(model).set_domain_engram_focus = SparkXLoopV03.set_domain_engram_focus
type(model).auto_detect_and_focus_domain = SparkXLoopV03.auto_detect_and_focus_domain
type(model).set_engram_flight_mode = SparkXLoopV03.set_engram_flight_mode
type(model).snapshot_engram_bank = SparkXLoopV03.snapshot_engram_bank
type(model).hot_swap_engram_bank = SparkXLoopV03.hot_swap_engram_bank

param_stats = count_en4b_a1b_params(model)
total_b = param_stats.get('model_total_params', param_stats.get('grand_total', 0)) / 1e9
active_b = param_stats.get('model_active_params', param_stats.get('active_total_per_token', 0)) / 1e9
rt_init = model.get_engram_routing_telemetry()
ppl_init = bench.evaluate_engram_perplexity_delta(model, tokenizer)

print('=' * 112)
print(f"[Loaded Checkpoint `{HF_REPO_ID}` Baseline Status]")
print(
    f"  Total Params : {total_b:.2f}B | Active/Tok: {active_b:.2f}B | "
    f"Active Experts: {rt_init['active_experts']} (Top-K={rt_init['active_top_k']}) | Mean Load CV: {rt_init['mean_load_cv']:.4f}"
)
print(
    f"  Held-Out PPL : 72L No-Engram = {ppl_init['no_engram_ppl']:.4f} -> 72L Full MoE-Engram = {ppl_init['with_engram_ppl']:.4f} "
    f"(Delta = {ppl_init['delta_ppl']:+.4f}, Next-Token Acc = {ppl_init['with_engram_acc']:.2f}%)"
)
print('=' * 112)

In [ ]:
# ==============================================================================
# Cell 3: Configure Continuation Mode, OpenCode Teacher & Dynamic MoE-Engram Settings
# ==============================================================================
from sparkx_loop_v03_train_env import auto_configure_vram_16k, build_default_cfg

# Choose Continuation Mode:
#   - 'SI_SRSI_AND_PHASE4'                : Fast & targeted (~15-25 min). Runs Phase-4 Multi-Domain Consolidation +
#                                           OpenCode-Supervised SI-SRSI Knowledge Engram Loop across Anti-Slop,
#                                           Web UX/Policy/AppSec, and STEM Reasoning.
#   - 'FULL_STREAMING_CORPUS_PLUS_SI_SRSI': Deep expansion (~1.5-3 hrs). Also streams 7-Pillar 2025-2026 Frontier
#                                           datasets and runs continued 8K/16K MoE-Engram-32 training before SI-SRSI.
CONTINUATION_MODE = 'SI_SRSI_AND_PHASE4'

# Stage A (Phase-4 Consolidation) & Stage B (SI-SRSI OpenCode Teacher Loop) Hyperparameters
PHASE4_CONSOLIDATION_STEPS        = 120
SI_SRSI_ROUNDS                    = 2
SI_SRSI_STEPS_PER_PROMPT          = 6
SI_SRSI_UNLIKELIHOOD_ALPHA        = 0.25   # Token-level unlikelihood penalty on banned Claude-ish/RLHF slop phrases
LR_ENGRAM                         = 1.5e-4
LR_GATES                          = 5.0e-4
LR_MHC                            = 3.0e-5
DEFAULT_FLIGHT_MODE_AFTER_TRAIN   = 'deep_knowledge'  # 'fast' (Top-4) | 'balanced' (Top-7) | 'deep_knowledge' (Top-16, 1.93B) | 'max_recall' (Top-24) | 'full_dense' (Top-32)
OPENCODE_TEACHER_MODEL            = os.environ.get('OPENCODE_MODEL', 'opencode/claude-sonnet-4-5')
PUSH_CONTINUED_MODEL_TO_HUB       = True

# Config for optional Deep Streaming Corpus Mode ('FULL_STREAMING_CORPUS_PLUS_SI_SRSI')
CFG = build_default_cfg()
CFG['HF_REPO_ID'] = HF_REPO_ID
CFG['CONTINUE_FROM_LOADED_HUB_MODEL'] = True
CFG['TARGET_TOTAL_SAMPLES'] = 2400
CFG['ENABLE_PHASE4_POST_TRAIN'] = True
CFG['PHASE4_STEPS'] = PHASE4_CONSOLIDATION_STEPS
CFG['ENABLE_SI_SRSI_ENGRAM_LOOP'] = True
CFG['SI_SRSI_ROUNDS'] = SI_SRSI_ROUNDS
CFG['SI_SRSI_CONSOLIDATION_STEPS_PER_PROMPT'] = SI_SRSI_STEPS_PER_PROMPT
CFG['SI_SRSI_UNLIKELIHOOD_ALPHA'] = SI_SRSI_UNLIKELIHOOD_ALPHA
CFG['OPENCODE_TEACHER_API_URL'] = OPENCODE_API_URL
CFG['OPENCODE_TEACHER_API_KEY'] = OPENCODE_API_KEY
CFG['OPENCODE_TEACHER_MODEL'] = OPENCODE_TEACHER_MODEL
CFG['ENG_DEFAULT_FLIGHT_MODE'] = DEFAULT_FLIGHT_MODE_AFTER_TRAIN
CFG['PUSH_TO_HUB'] = PUSH_CONTINUED_MODEL_TO_HUB
CFG = auto_configure_vram_16k(CFG)

# Override STAGES so Deep Streaming Corpus Mode only runs 32-Expert Top-7 Continuation Stages (skipping Step-0 Stage 1/2)
CFG['STAGES'] = [
    {
        'name': 'Stage3_Continued_Engram32_Top7_Absorption',
        'seq_len': 8192,
        'steps': 90,
        'passes': 2,
        'start': 4,
        'end': 28,
        'enable_p3': True,
        'p3_start': 12,
        'p3_end': 24,
        'use_mhc': True,
        'use_engram': True,
        'freeze_backbone': True,
        'freeze_engram': False,
        'upcycle_8_to_32': False,
        'loss_floor': 0.28,
    },
    {
        'name': 'Stage4_Continued_Joint_FullFT_72L_Engram32',
        'seq_len': 16384,
        'steps': 60,
        'passes': 2,
        'start': 4,
        'end': 28,
        'enable_p3': True,
        'p3_start': 12,
        'p3_end': 24,
        'use_mhc': True,
        'use_engram': True,
        'freeze_backbone': False,
        'backbone_lr_mult': 0.15,
        'freeze_engram': False,
        'upcycle_8_to_32': False,
        'loss_floor': 0.26,
    },
]
print(f"[Continuation Configured] Mode={CONTINUATION_MODE} | FlightMode={DEFAULT_FLIGHT_MODE_AFTER_TRAIN} | Teacher={OPENCODE_TEACHER_MODEL}")

In [ ]:
# ==============================================================================
# Cell 4: (Optional Deep Corpus Mode) Stream 7-Pillar Frontier Datasets & Run
#         Continued 8K/16K MoE-Engram-32 Training on Top of `TassP0lar/Spark-X2.6-4B-En4B-A1B`
#   - Automatically reuses `STAGE_POOLS` if already tokenized in RAM!
# ==============================================================================
import gc, importlib, subprocess, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core
import sparkx_loop_v03_train_env
import sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
importlib.reload(sparkx_loop_v03_benchmarks)

from sparkx_loop_v03_train_env import (
    build_16k_stage_pools,
    curate_all_pillars_v03,
    generate_opencode_go_trajectories,
    run_v03_four_stage_training,
)

if CONTINUATION_MODE == 'FULL_STREAMING_CORPUS_PLUS_SI_SRSI':
    # Ensure 72L passes=2 (4 + 2*24 + 12 + 8 = 72L) and activation checkpointing are active
    for st in CFG.get('STAGES', []):
        if st.get('enable_p3', False) and st.get('passes', 2) > 2:
            st['passes'] = 2
    model.enable_checkpoint = True
    model.settings.checkpoint_layers = True
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if 'STAGE_POOLS' not in globals() or not STAGE_POOLS:
        curated_records = curate_all_pillars_v03(CFG)
        teacher_records = generate_opencode_go_trajectories(CFG, api_key=OPENCODE_API_KEY)
        STAGE_POOLS = build_16k_stage_pools(curated_records + teacher_records, tokenizer, CFG)
    else:
        print(f"[RAM Cache Hit] Reusing already-packed STAGE_POOLS ({ {k: len(v) for k, v in STAGE_POOLS.items()} }).")

    continued_train_summary = run_v03_four_stage_training(
        model=model,
        tokenizer=tokenizer,
        stage_pools=STAGE_POOLS,
        cfg=CFG,
        hf_token=HF_TOKEN,
    )
else:
    print("[Fast Continuation Mode] Skipping multi-hour streaming pool; proceeding directly to Stage A (Phase-4) + Stage B (SI-SRSI) in Cell 5.")

In [ ]:
# ==============================================================================
# Cell 5: Execute Stage A (Phase-4 Consolidation) + Stage B (`SI-SRSI` OpenCode-Supervised
#         Dynamic MoE-Engram Knowledge Formation) + Stage C (On-The-Fly Flight Mode Ablation)
#   - Automatically runs if Phase-4 or SI-SRSI was not yet completed in Cell 4, and pushes
#     `loop_v03_state.pt`, `loop_config_v03.json`, and `si_srsi_engram_report.json`
#     to `https://huggingface.co/TassP0lar/Spark-X2.6-4B-En4B-A1B`
# ==============================================================================
import importlib, subprocess
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core
import sparkx_loop_v03_benchmarks as bench
importlib.reload(sparkx_loop_v03_core)
importlib.reload(bench)

_prev = globals().get('continued_train_summary', {}) or {}
need_p4 = _prev.get('phase4_report') is None
need_si = _prev.get('si_srsi_report') is None

if CONTINUATION_MODE == 'SI_SRSI_AND_PHASE4' or need_p4 or need_si:
    continuation_report = bench.run_continue_from_hub_engram_expansion(
        model=model,
        tokenizer=tokenizer,
        num_phase4_steps=PHASE4_CONSOLIDATION_STEPS if need_p4 else 0,
        num_si_srsi_rounds=SI_SRSI_ROUNDS if need_si else 0,
        consolidation_steps_per_prompt=SI_SRSI_STEPS_PER_PROMPT,
        lr_engram=LR_ENGRAM,
        lr_gates=LR_GATES,
        lr_mhc=LR_MHC,
        unlikelihood_alpha=SI_SRSI_UNLIKELIHOOD_ALPHA,
        default_flight_mode=DEFAULT_FLIGHT_MODE_AFTER_TRAIN,
        opencode_api_url=OPENCODE_API_URL,
        opencode_api_key=OPENCODE_API_KEY,
        opencode_model=OPENCODE_TEACHER_MODEL,
        save_dir='/content/sparkx_v03_continued_hub',
        push_to_hub=PUSH_CONTINUED_MODEL_TO_HUB and bool(HF_TOKEN),
        hub_repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
else:
    print("[Info] Phase-4 + SI-SRSI already executed at the end of Cell 4 (`run_v03_four_stage_training`).")
    flight_ablation = bench.evaluate_dynamic_engram_flight_modes(model, tokenizer)

In [ ]:
# ==============================================================================
# Cell 6: Free VRAM & Unused Checkpoints -> Run Calibrated SI-SRSI Loop ->
#         Interactive 3-Domain Knowledge Engram Audit (`anti_slop`, `web_ux_sec`, `stem_reason`)
# ==============================================================================
import gc, importlib, subprocess, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core
import sparkx_loop_v03_benchmarks as bench
importlib.reload(sparkx_loop_v03_core)
importlib.reload(bench)

# 0. Free RAM/VRAM & prune redundant intermediate checkpoints on disk
for _tmp_var in ('STAGE_POOLS', 'curated_records', 'teacher_records'):
    if _tmp_var in globals():
        globals()[_tmp_var] = None
bench.free_vram_and_prune_checkpoints(model=model, keep_latest_n=1, prune_intermediate_stage_dirs=True)

# 1. Run calibrated SI-SRSI (with max_new_tokens=1024 + think_budget_tokens=192 & 1:1 Phase-4 anchor interleaving)
si_srsi_report = bench.run_si_srsi_engram_loop(
    model=model,
    tokenizer=tokenizer,
    num_rounds=2,
    steps_per_round=20,
    lr_engram=4.0e-5,
    lr_gates=1.0e-4,
    lr_mhc=1.0e-5,
    ftpo_slop_weight=0.15,
    opencode_api_url=OPENCODE_API_URL,
    opencode_api_key=OPENCODE_API_KEY,
    opencode_model=OPENCODE_TEACHER_MODEL,
    save_dir='/content/sparkx_v03_continued_hub',
    push_to_hub=PUSH_CONTINUED_MODEL_TO_HUB and bool(HF_TOKEN),
    hub_repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)

# 2. Interactive 3-Domain Generation Audit (`anti_slop`, `web_ux_sec`, `stem_reason`)
test_prompts = [
    (
        'anti_slop',
        'Write an engineering post-mortem report analyzing why our distributed Redis cache experienced a 140ms p99 latency spike during shard rebalancing, and specify the concrete remediation steps.'
    ),
    (
        'web_ux_sec',
        'Design a full-stack API key management module (HTML/CSS/JS frontend + Node/Express or Python/FastAPI backend) following strict UX policy (loading/empty/error/success states, WCAG 2.2 AA accessibility) and defensive application security (CSP, CSRF protection, HttpOnly SameSite=Strict cookies, parameterized SQL, and rate limiting).'
    ),
    (
        'stem_reason',
        'Find all positive integers $n$ such that $n^4 + n^2 + 1$ is a prime number, and prove why no larger positive integer can satisfy this condition. Conclude with \\boxed{1}.'
    ),
]

for dom, prompt in test_prompts:
    focus_info = model.set_domain_engram_focus(dom, boost=0.08)
    resp = bench.generate_chat_response(
        model, tokenizer,
        user_prompt=prompt,
        max_new_tokens=1024,
        think_budget_tokens=192,
        temperature=0.1,
    )
    if dom == 'anti_slop':
        sc = bench.score_anti_slop_composition(resp['answer_text'])
    elif dom == 'web_ux_sec':
        sc = bench.score_web_ux_and_defensive_security(resp['answer_text'])
    else:
        sc = bench.score_stem_knowledge_and_reasoning(resp['raw_text'], '1')
    print('=' * 112)
    print(f"[Domain Focus: `{dom}` | Zones={focus_info['zone_experts']} | Score={sc['score']:.1f}/100]")
    print(f"Answer Preview (first 450 chars):\n{resp['answer_text'][:450]}...")

model.set_domain_engram_focus('general', boost=0.0)
bench.free_vram_and_prune_checkpoints(model=model, keep_latest_n=1, prune_intermediate_stage_dirs=True)

In [ ]:
# ==============================================================================
# Cell 7: Post-Continuation 11-Benchmark Frontier Evaluation Suite
#   - Evaluates `IFEval`, `IFBench`, `GPQA Diamond`, `SuperGPQA (285 Disciplines)`,
#     `HLE`, `SWE-bench Pro`, `SWE-bench Verified`, `HumanEval`, `LiveCodeBench`,
#     `MATH-500`, and `AIME 2024/2025 + Omni-MATH`
# ==============================================================================
import importlib
import sparkx_loop_v03_benchmarks as bench
importlib.reload(bench)

frontier_report = bench.run_full_frontier_benchmark_suite(
    model=model,
    tokenizer=tokenizer,
    flight_mode=DEFAULT_FLIGHT_MODE_AFTER_TRAIN,
    run_dynamic_flight_validation=True,
    samples_ifeval=25,
    samples_ifbench=25,
    samples_gpqa=25,
    samples_supergpqa=20,
    samples_hle=15,
    samples_swe_pro=10,
    samples_swe_verified=10,
    samples_humaneval=25,
    samples_livecode=15,
    samples_math500=25,
    samples_aime_omni=15,
    output_dir='/content/sparkx_v03_continued_benchmark_results',
    token=HF_TOKEN,
)